# Toolkit d'investissement — DCF, Optimisation de portefeuille & Suivi des transactions du Congrès

Ce notebook fusionne et corrige `DCF.ipynb` et `MVO_1_.ipynb`, et ajoute un module
de suivi des transactions boursières déclarées par les membres du Congrès américain.

Le code métier vit dans 3 modules Python importés ci-dessous (à garder à côté de ce
notebook, ou à réutiliser directement dans votre terminal d'intelligence marchés) :

- `dcf_model.py` — valorisation DCF, WACC/CAPM, Gordon-Shapiro, ratios financiers, Piotroski F-score
- `portfolio_optimization.py` — Monte Carlo, frontière efficiente (scipy/cvxpy), HRP, backtest
- `congress_trading.py` — transactions déclarées des sénateurs/représentants (STOCK Act)

**Résumé des corrections apportées** (détail en commentaire en tête de chaque module) :
1. Bugs bloquants corrigés : `IndentationError` sur la recommandation Buy/Sell, `.dropna()`
   appelé sur une liste Python au lieu d'une Series, `get_efficient_frontier` qui ignorait
   ses propres paramètres et lisait des variables globales.
2. Biais de calcul corrigés : coût de la dette négatif (signe de `Interest Expense`),
   `np.isnan(None)` qui plantait, croissance du FCF par moyenne arithmétique (biaisée)
   remplacée par un CAGR.
3. Fragilité supprimée : scraping web de SimplyWallSt (souvent bloqué) remplacé par le
   CAGR de l'ETF sectoriel déjà utilisé ailleurs dans vos notebooks ; dépendance à
   `pyfolio` (non maintenu) remplacée par des métriques de backtest calculées à la main.
4. Doublons fusionnés : deux fonctions DCF quasi identiques, deux implémentations
   partielles du Piotroski F-score, deux calculateurs de ratios redondants.
5. Nouveau : suivi des transactions du Congrès (`congress_trading.py`), avec agrégation
   par ticker et détection des nouvelles transactions pour du monitoring récurrent.
6. **Point d'attention non résolu (biais de look-ahead)** : `yfinance.Ticker.info` reflète
   l'état ACTUEL du marché (beta, prix, capitalisation), pas l'état à la date des
   états financiers historiques utilisés pour projeter le FCF. Pour un vrai usage dans
   un terminal exigeant l'absence de look-ahead bias, il faudrait une source de données
   fondamentales point-in-time (yfinance n'en fournit pas) — voir `walk_forward_backtest`
   dans `portfolio_optimization.py` pour un exemple de séparation stricte
   estimation / test sur le volet portefeuille.


In [ ]:
import sys
sys.path.append('.')  # adaptez si les modules sont ailleurs

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import dcf_model as dcf
import portfolio_optimization as po
import congress_trading as ct

WATCHLIST = ["AAPL", "MSFT", "GOOG", "AMZN", "META", "NVDA"]


## 1. Valorisation DCF

Valorisation intrinsèque + ratios financiers pour un ticker donné.

In [ ]:
ticker = "AAPL"

report = dcf.run_dcf(ticker, projection_years=5)
print(report.summary())


In [ ]:
ratios = dcf.compute_financial_ratios(ticker)
pd.Series(ratios)


In [ ]:
# Comparaison de la valeur intrinsèque à l'historique de prix
prices = po.download_prices([ticker], years=5)[ticker]

plt.figure(figsize=(10, 5))
plt.plot(prices, label="Prix de marché")
plt.axhline(report.intrinsic_value, color="r", linestyle="--", label="Valeur intrinsèque (DCF)")
plt.axhline(report.last_price, color="g", linestyle="--", label="Dernier prix")
plt.title(f"{ticker} — Prix de marché vs valeur intrinsèque DCF")
plt.ylabel("Prix ($)")
plt.legend()
plt.tight_layout()
plt.show()


### Passage à l'échelle sur une watchlist

Applique le DCF à toute une liste de tickers pour un premier écrémage.
Chaque appel est protégé (`try/except`) pour qu'un ticker sans données
fondamentales suffisantes n'interrompe pas toute la boucle.

In [ ]:
dcf_results = []
for tk in WATCHLIST:
    try:
        r = dcf.run_dcf(tk)
        dcf_results.append({
            "ticker": tk, "price": r.last_price, "intrinsic_value": r.intrinsic_value,
            "margin_of_safety": r.margin_of_safety, "recommendation": r.recommendation(),
        })
    except Exception as e:
        print(f"{tk}: DCF impossible ({e})")

dcf_df = pd.DataFrame(dcf_results).set_index("ticker").sort_values("margin_of_safety", ascending=False)
dcf_df


## 2. Optimisation de portefeuille (MVO / frontière efficiente / HRP)

In [ ]:
prices_basket = po.download_prices(WATCHLIST, years=3)
returns, avg_returns, cov_mat = po.compute_annualized_stats(prices_basket)

ew_returns = po.equal_weight_returns(returns)
print("Backtest équipondéré (in-sample) :")
for k, v in po.backtest_metrics(ew_returns).items():
    print(f"  {k:20}: {v:.3f}")


In [ ]:
mc_df, mc_weights = po.monte_carlo_frontier(avg_returns, cov_mat, n_portfolios=20_000)

target_returns = np.linspace(mc_df["returns"].min(), mc_df["returns"].max(), 40)
frontier = po.efficient_frontier_scipy(avg_returns, cov_mat, target_returns)
frontier_vols = [f["volatility"] for f in frontier]

max_sharpe = po.max_sharpe_scipy(avg_returns, cov_mat)
min_vol = po.min_volatility_scipy(avg_returns, cov_mat)

plt.figure(figsize=(10, 6))
sc = plt.scatter(mc_df["volatility"], mc_df["returns"], c=mc_df["sharpe_ratio"], cmap="RdYlGn", alpha=0.4, s=8)
plt.colorbar(sc, label="Sharpe ratio")
plt.plot(frontier_vols, target_returns, "b--", linewidth=2, label="Frontière efficiente")
plt.scatter([max_sharpe["volatility"]], [max_sharpe["return"]], color="black", marker="*", s=250, label="Sharpe max")
plt.scatter([min_vol["volatility"]], [min_vol["return"]], color="blue", marker="P", s=150, label="Volatilité min")
plt.xlabel("Volatilité annualisée")
plt.ylabel("Rendement annualisé")
plt.title("Frontière efficiente — " + ", ".join(WATCHLIST))
plt.legend()
plt.tight_layout()
plt.show()

print("Portefeuille Sharpe max :")
for tk, w in zip(WATCHLIST, max_sharpe["weights"]):
    print(f"  {tk:6}: {w:.1%}")


### Méthodes optionnelles (dépendances lourdes)

`hrp_optimize` et `max_sharpe_pypfopt` nécessitent `PyPortfolioOpt` ;
`efficient_frontier_cvxpy` nécessite `cvxpy`. Elles lèvent une erreur explicite
si le paquet n'est pas installé plutôt que de faire planter l'import du module.

In [ ]:
try:
    hrp = po.hrp_optimize(prices_basket)
    print("HRP weights:", hrp["weights"])
except ImportError as e:
    print(e)


### Backtest hors-échantillon (sans biais de look-ahead)

`walk_forward_backtest` ré-optimise les poids périodiquement en n'utilisant que
les rendements PASSÉS à chaque date de rebalancement — contrairement aux exemples
du notebook original où les poids optimaux étaient calculés et appliqués sur la
même fenêtre de données.

In [ ]:
wf_returns = po.walk_forward_backtest(
    prices_basket,
    optimizer_fn=lambda a, c: po.max_sharpe_scipy(a, c)["weights"],
    lookback_days=252,
    rebalance_every=21,
)
print("Backtest walk-forward (hors-échantillon) :")
for k, v in po.backtest_metrics(wf_returns).items():
    print(f"  {k:20}: {v:.3f}")


## 3. Suivi des transactions boursières du Congrès américain

Nécessite une clé API gratuite (750 crédits offerts, sans carte bancaire) :
https://www.disclosedcapitol.com/developers — à passer via la variable
d'environnement `DC_API_KEY` ou directement à `DisclosedCapitolClient(api_key=...)`.


In [ ]:
import os
# os.environ["DC_API_KEY"] = "votre_cle_ici"

client = ct.DisclosedCapitolClient()  # lit DC_API_KEY dans l'environnement

trades = []
for tk in WATCHLIST:
    trades.extend(client.recent_trades(ticker=tk, limit=25))

trades_df = ct.trades_to_dataframe(trades)
trades_df.sort_values("transaction_date", ascending=False).head(15)


In [ ]:
sentiment = ct.ticker_sentiment(trades_df)
sentiment


### Monitoring récurrent (nouvelles transactions uniquement)

`TradeMonitor` conserve un état local (`congress_trades_seen.json`) pour ne
remonter, aux exécutions suivantes, que les transactions jamais vues — utile en
tâche planifiée plutôt qu'en exécution manuelle répétée.

In [ ]:
monitor = ct.TradeMonitor(state_path="congress_trades_seen.json")
new_trades = monitor.watch(client, WATCHLIST)
print(f"{len(new_trades)} nouvelle(s) transaction(s) depuis le dernier passage.")
new_trades


## 4. Vue combinée : DCF + Optimisation + Signal Congrès

Croise les trois sources sur la watchlist : sous/sur-évaluation DCF, poids dans
le portefeuille Sharpe max, et activité récente des membres du Congrès sur
chaque titre. **Le signal Congrès est un indicateur de contexte, pas une
recommandation** — le décalage de déclaration (jusqu'à 45 jours légalement)
signifie qu'une transaction "récente" a en réalité pu être exécutée bien plus tôt.

In [ ]:
combined = dcf_df.copy()
combined["max_sharpe_weight"] = pd.Series(dict(zip(WATCHLIST, max_sharpe["weights"])))

if not trades_df.empty:
    sentiment_reindexed = sentiment.reindex(combined.index)
    combined["congress_net_trades"] = sentiment_reindexed["n_buys"].fillna(0) - sentiment_reindexed["n_sells"].fillna(0)
    combined["congress_last_trade"] = sentiment_reindexed["last_transaction"]
else:
    combined["congress_net_trades"] = 0

combined.sort_values("margin_of_safety", ascending=False)
